# Run and plot the reproduction
This notebook only configures and calls package APIs. Experiment logic belongs in `src/sentiment_manifold`.

In [ ]:
# In a fresh Colab runtime, uncomment after cloning or uploading the project.
# %pip install -e '.[notebooks]'
from pathlib import Path
from sentiment_manifold.config import ReproductionConfig
from sentiment_manifold.storage import maybe_mount_google_drive, resolve_output_dir

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
USE_GOOGLE_DRIVE = False
maybe_mount_google_drive(USE_GOOGLE_DRIVE)
output_dir = resolve_output_dir(use_google_drive=USE_GOOGLE_DRIVE)
cfg = ReproductionConfig.load(PROJECT_ROOT / 'configs/reproduction.yaml')
cfg.experiment.output_dir = str(output_dir)
output_dir

In [ ]:
# Start with a cheap smoke run. Remove these overrides for the full sweep.
cfg.experiment.layers = [0, 6, 12] if cfg.model.name == 'gpt2-small' else [0, 14, 28]
cfg.das.epochs = 1
cfg.data.sst_max_pairs = 8
cfg.experiment.openwebtext_resample_ablation = False

In [ ]:
RUN_EXPERIMENT = False  # Set True after reviewing the configuration.
if RUN_EXPERIMENT:
    from sentiment_manifold.experiment import run_reproduction
    run_dir = run_reproduction(cfg)
    print(run_dir)

In [ ]:
# Plot any completed run.
from sentiment_manifold.plotting import plot_run
candidate = output_dir / cfg.model.name
if candidate.exists() and (candidate / 'metrics.csv').exists():
    display(*[str(path) for path in plot_run(candidate)])